# Credit Risk Analyzer

In [41]:
import numpy as np
import pandas as pd
import locale
import os

## 0. Input Validation

In [42]:
def get_input(prompt, cast, min_val, max_val, error_msg):
    value = cast(input(prompt))
    while value < min_val or value > max_val:
        print(error_msg)
        value = cast(input(prompt))
    return value

## 0.1 System Currency

In [43]:
def get_currency_symbol():
    locale.setlocale(locale.LC_ALL, '')  # Use system configuration
    conv = locale.localeconv()
    return conv['currency_symbol']

symbol = get_currency_symbol()

## 1. Data Collection

In [44]:
def collect_customer_data():
    print(f'\n{'=' * 60}\n{'CREDIT RISK ANALYZER'.center(60)}\n{'=' * 60}')
    
    name = input('Customer name: ')
    score = get_input(f'Credit score (0 to 1000): ',                  float, 0,  1000, 'Invalid. Must be between 0 and 1000.')
    income = get_input(f'Monthly income ({symbol}): ',                float, 0,  1e18, 'Invalid. Must be greater than 0.')
    debt = get_input(f'Total current debt ({symbol}): ',              float, 0,  1e18, 'Invalid. Must be greater than 0.')
    delays = get_input('Delays in the last 24 months: ',              int,   0,  24,   'Invalid. Must be between 0 and 24.')
    time_bank = get_input('Relationship with the bank (months): ',    int,   1,  1e18, 'Invalid. Must be greater than 0.')
    requests = get_input('Loan applications in the last 6 months: ',  int,   0,  6,    'Invalid. Must be between 0 and 6.')

    customer = {
        'Name': name,
        'Score': score,
        'Income': income,
        'Debt': debt,
        'Delays': delays, 
        'Time_bank': time_bank,
        'Requests': requests
    }
    return customer

## 2. Normalization

In [45]:
def normalize_customer(customer):
    
    score_norm = customer['Score'] / 1000 
    income_norm = np.clip(customer['Income'] / 20000, 0, 1)
    
    debt_ratio = customer['Debt'] / (customer['Income'] + 1)
    debt_norm = 1 - np.clip(debt_ratio / 10, 0, 1)

    delays_norm = np.clip(1 - customer['Delays'] / 5, 0, 1)
    time_bank_norm = np.clip(customer['Time_bank'] / 60, 0, 1)
    requests_norm = np.clip(1 - customer['Requests'] / 6, 0, 1)

    return np.array([score_norm, income_norm, debt_norm,
                     delays_norm, time_bank_norm, requests_norm])

## 3. Weights and Final Score

In [46]:
def calculate_score(normalized):

    weights = np.array([
        0.30,  # Score
        0.25,  # Income
        0.20,  # Debt
        0.10,  # Delays
        0.10,  # Time_bank
        0.05   # Requests
    ])

    final_score = np.dot(normalized, weights)

    return final_score

## 4. Probability of Default + Classification

In [47]:
def classify_customer(final_score):

    prob_default = 1 - final_score

    if prob_default < 0.05:
        rating = 'AAA'
        action = 'Automatic approval'
    elif prob_default < 0.15:
        rating = 'A'
        action = 'Simplified review'
    elif prob_default < 0.30:
        rating = 'B'
        action = 'Managerial review'
    elif prob_default < 0.50:
        rating = 'C'
        action = 'Conditional approval'
    else:
        rating = 'D'
        action = 'Recommended refusal'

    return prob_default, rating, action

## 5. Final Report

In [48]:
def generate_report(customer, normalized, score, prob_default, rating, action):

    details = pd.DataFrame({
        'Variable': ['Score', 'Income', 'Debt Ratio', 'Delays', 'Time w/ Bank', 'Requests'],
        'Raw Value': [customer['Score'], customer['Income'], customer['Debt'],
                      customer['Delays'], customer['Time_bank'], customer['Requests']],
        'Normalized (0-1)': normalized.round(4)
    })

    print(f'{'Costumer':.<30} {customer['Name']}')
    print(f'{'Final Score':.<30} {score:.4f}')
    print(f'{'P(Default)':.<30} {prob_default:.2%}')
    print(f'{'Rating':.<30} {rating}')
    print(f'{'Action':.<30} {action}')
    print(f'{'=' * 60}')

    print(f'\n{'=' * 60}\n{'VARIABLE DETAILS'.center(60)}\n{'=' * 60}')
    print(f'{'Variable':<15}{'Raw Value':>15}{'Normalized (0-1)':>20}')

    for var, raw, norm in zip(details['Variable'], details['Raw Value'], details['Normalized (0-1)']):
        print(f'{var:<15}{raw:>15.2f}{norm:>20.4f}')
    
    print(f'{'=' * 60}')

In [49]:
def run_analyzer():
    customer = collect_customer_data()
    normalized = normalize_customer(customer)
    score = calculate_score(normalized)
    prob_default, rating, action = classify_customer(score)

    generate_report(customer, normalized, score, prob_default, rating, action)
    
    results = pd.DataFrame([{
        'Name':        customer['Name'],
        'Final Score': round(score, 4),
        'P(Default)':  round(prob_default, 4),
        'Rating':      rating,
        'Action':      action
    }])
    
    file_exists = os.path.exists('results.csv')
    if file_exists:
        results.to_csv('results.csv', mode='a', header=False, index=False)
    else:
        results.to_csv('results.csv', index=False)
    print('\n✓ Result saved to results.csv')

run_analyzer()


                    CREDIT RISK ANALYZER                    
Costumer...................... Pedro
Final Score................... 0.8615
P(Default).................... 13.85%
Rating........................ A
Action........................ Simplified review

                      VARIABLE DETAILS                      
Variable             Raw Value    Normalized (0-1)
Score                   879.30              0.8793
Income                12903.00              0.6452
Debt Ratio             3400.00              0.9737
Delays                    0.00              1.0000
Time w/ Bank             93.00              1.0000
Requests                  1.00              0.8333

✓ Result saved to results.csv
